# Day 32


In [4]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)

assert os.getenv("GEMINI_API_KEY"), "Gemini API key is missing."

print("Gemini API key is configured.")


Gemini API key is configured.


In [5]:
from langchain.chat_models import init_chat_model

In [6]:
model = init_chat_model('google_genai:gemini-3.5-flash-lite',temperature=0.3,max_retries=2)

In [7]:
response = model.invoke('hello')

c:\My Files\LTM-LAB\venv\Lib\site-packages\langchain_google_genai\chat_models.py:4109: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [8]:
response

AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EpYBCpMBAWkUfRNRlZcIwcJJxGnQLiUGo2EHQj6jePM5/izQs2uekEL/m/UJM3NB12GEgzNpEezNimcWNej/Xs+sAXixUNDaWNEgSze3jbmzXKHAZY6VRACf657uh2DIhKNBWU8Ck+yAJF3TaEmTvapjwuz40vDD4ApNNbxJgp3rr0vvFzw5GtySJC0sa7/THt00rRr0CiKU'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'stop', 'model_provider': 'google_genai'}, id='lc_run--01a2166d-df29-7231-a302-5413d0f9a696-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 9, 'total_tokens': 11, 'input_token_details': {'cache_read': 0}})

In [9]:
response.text

'Hello! How can I help you today?'

In [10]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

In [11]:
messages = [
    SystemMessage(
        content="You are an insurance customer support assistant. "
                "Explain insurance concepts in simple language."
    ),

    HumanMessage(
        content="What is an insurance premium? tell me in 2 lines"
    )
]

response = model.invoke(messages)

print(response.content)

[{'type': 'text', 'text': 'An insurance premium is simply the regular payment—like a monthly or yearly bill—that you pay to keep your insurance active. Think of it as the price you pay for your peace of protection.', 'extras': {'signature': 'EpYBCpMBAWkUfRNMrUpxUnTuC4IqI8JHRRxo0NFCYVWo9AgDd+L3pm/JRswq89kSnIb0tWIjDuUmZGadRK6GMfZO+paf0+i3aIe252NfTP4W3UUspPRDG1vusi/iXOu9v4YpGYs/JYbCtO6abActfurAz3si8bmo1PG6lQ4eLyZ8eNck50v+GIrHPJWxyumSnw8g9ATBKnQG'}}]


In [12]:
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate,MessagesPlaceholder

In [13]:
insurance_prompt = PromptTemplate.from_template(
    '''
    You are an insurance support assistant.

    Explain the following insurance concept in 1 line, using a practical example.

    Question : {question}
    '''
)

In [14]:
print(insurance_prompt)

input_variables=['question'] input_types={} partial_variables={} template='\n    You are an insurance support assistant.\n\n    Explain the following insurance concept in 1 line, using a practical example.\n\n    Question : {question}\n    '


In [15]:
formatted_prompt = insurance_prompt.format(
    question='What is an insurance deductible'
)

print(formatted_prompt)


    You are an insurance support assistant.

    Explain the following insurance concept in 1 line, using a practical example.

    Question : What is an insurance deductible
    


In [16]:
response = model.invoke(formatted_prompt)

response.text

'An insurance deductible is the fixed amount you must pay out-of-pocket before your insurance covers the rest; for example, if you have a $500 deductible and face a $2,000 car repair bill, you pay the first $500 and your insurer pays the remaining $1,500.'

In [17]:
chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an insurance support assistant. "
            "Explain insurance concepts in simple language."
        ),
        (
            "human",
            "{question}"
        )
    ]
)

In [18]:
formatted_messages = chat_prompt.format_messages(
    question='What is the insurance waikting period'
)

print(formatted_messages)

[SystemMessage(content='You are an insurance support assistant. Explain insurance concepts in simple language.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the insurance waikting period', additional_kwargs={}, response_metadata={})]


In [19]:
response = model.invoke(formatted_messages)
print(response.text)

Think of an insurance **waiting period** as a **"holding pattern"** or a **trial period** before your benefits actually kick in. 

During this time, you have the insurance policy, but the insurance company won't pay out for certain claims yet. 

Here is a simple way to understand how it works in different types of insurance:

### 1. Health Insurance
* **What it means:** Sometimes, health insurance plans make you wait a certain amount of time before they cover specific treatments or pre-existing conditions.
* **Example:** You buy a new health insurance policy, and it has a 12-month waiting period for maternity care. If you have a baby in month 6, the insurance won't pay for it. But if you have a baby in month 14, it's covered.

### 2. Disability Insurance (Income Protection)
* **What it means:** If you get hurt or sick and can't work, this is the time you must wait between getting sick/injured and when the insurance company starts sending you checks. 
* **Example:** Your policy has a 30

In [20]:
insurance_support_prompt = chat_prompt.partial(
    assistant_role="Insurance Support Specialist",
    communication_style="simple, friendly and professional"
)

messages = insurance_support_prompt.format_messages(
    question="What is a waiting period?"
)

response = model.invoke(messages)

print(response.text)

Think of a **waiting period** (sometimes called an "elimination period") as a **buffer time** or a **countdown** you have to sit through before your insurance starts paying out. 

Here is the easiest way to understand it:

### 1. The Waiting Period for *Buying* Insurance
Imagine you buy a new health or dental insurance policy. The insurance company might say, *"You have to wait 30 days before we cover routine check-ups."* 
* During those 30 days, you pay your monthly bill (premium), but if you go to the dentist, you have to pay out of your own pocket. 
* Once the 30 days are up, the insurance kicks in. 
* **Why do they do this?** It stops people from waiting until they are already sick or hurt to buy insurance, use it immediately, and then cancel it.

### 2. The Waiting Period for *Using* a Benefit (Like Disability Insurance)
Imagine you have disability insurance, which replaces your paycheck if you get hurt and can't work. Your policy might have a **90-day waiting period**. 
* You bre

In [ ]:
chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an insurance support assistant. "
            "Use the conversation history to understand "
            "the customer's question."
        ),

        MessagesPlaceholder(
            variable_name="history"
        ),

        (
            "human",
            "{question}"
        )
    ]
)

In [24]:
from langchain_core.messages import HumanMessage, AIMessage

conversation_history = [
    HumanMessage(
        content="I'm looking for health insurance."
    ),
    AIMessage(
        content="Certainly. Are you looking for an individual or family plan?"
    ),
    HumanMessage(
        content="I'm 30 years old and looking for an individual plan."
    ),
    AIMessage(
        content="Understood. You're looking for an individual health insurance plan."
    )
]

In [22]:
response = model.invoke(formatted_messages)

In [23]:
print(response.text)

Think of an insurance **waiting period** as a **"cooling-off" or "not yet ready" period**. 

It is a specific amount of time you have to wait after buying a policy before your insurance coverage actually starts paying for certain things. 

Here is how it works in plain English:

### Why do insurance companies use waiting periods?
Insurance is meant to cover unexpected events. Without waiting periods, people might only buy insurance *after* they already know they need it (like buying health insurance the day before a scheduled surgery, or buying car insurance right after a storm is predicted). Waiting periods help keep insurance fair and affordable for everyone.

### Common Examples:

*   **Health Insurance:** You might have a waiting period for specific treatments, maternity care, or pre-existing conditions. During this time, you pay your monthly bill, but the insurance won’t cover those specific costs yet.
*   **Disability Insurance:** If you get hurt and can’t work, there is often a 

In [ ]:
from langchain_core.output_parsers import StrOutputParser,ListOutputParser

In [27]:
response = model.invoke('Explain the difference between a health insurance premium and deductable')

print(type(response))
print(response.content)

c:\My Files\LTM-LAB\venv\Lib\site-packages\langchain_google_genai\chat_models.py:4109: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


<class 'langchain_core.messages.ai.AIMessage'>
[{'type': 'text', 'text': 'Think of your health insurance like a car insurance policy. You pay just to have it, but you also pay out of pocket before the insurance really kicks in. \n\nHere is the easiest way to understand the difference between a **premium** and a **deductible**:\n\n---\n\n### 1. The Premium (The "Subscription" Fee)\n* **What it is:** The amount you pay to your insurance company **every month** just to keep your health plan active, whether you go to the doctor or not. \n* **Analogy:** It’s just like Netflix, Spotify, or your gym membership. You pay this fee every month to maintain access to the service.\n* **Who pays it:** Usually, you and/or your employer. If you get insurance through your job, the premium is often automatically deducted from your paycheck.\n* **Does it count toward your medical care?** No. Paying your premium doesn\'t mean your medical bills are paid; it just means you *have* insurance.\n\n### 2. The De

In [28]:
parser = StrOutputParser()

text_response = parser.invoke(response)

print(type(text_response))
print(text_response)

<class 'langchain_core.messages.base.TextAccessor'>
Think of your health insurance like a car insurance policy. You pay just to have it, but you also pay out of pocket before the insurance really kicks in. 

Here is the easiest way to understand the difference between a **premium** and a **deductible**:

---

### 1. The Premium (The "Subscription" Fee)
* **What it is:** The amount you pay to your insurance company **every month** just to keep your health plan active, whether you go to the doctor or not. 
* **Analogy:** It’s just like Netflix, Spotify, or your gym membership. You pay this fee every month to maintain access to the service.
* **Who pays it:** Usually, you and/or your employer. If you get insurance through your job, the premium is often automatically deducted from your paycheck.
* **Does it count toward your medical care?** No. Paying your premium doesn't mean your medical bills are paid; it just means you *have* insurance.

### 2. The Deductible (The "Threshold" Amount)
*

In [29]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

list_parser = CommaSeparatedListOutputParser()

format_instructions = list_parser.get_format_instructions()

print(format_instructions)

from langchain_core.prompts import ChatPromptTemplate

list_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an insurance documentation assistant."
        ),
        (
            "human",
            """
            List five documents commonly required
            when submitting a motor insurance claim.

            {format_instructions}
            """
        )
    ]
)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [30]:
list_chain = (
    list_prompt
    | model
    | list_parser
)

result = list_chain.invoke(
    {
        "format_instructions": format_instructions
    }
)

print(result)
print(type(result))

c:\My Files\LTM-LAB\venv\Lib\site-packages\langchain_google_genai\chat_models.py:4109: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


['Completed claim form', "Driver's driving license", 'Police report', 'Vehicle registration certificate', 'Photographs of the damage']
<class 'list'>


In [31]:
from pydantic import BaseModel, Field
from typing import Literal

class QueryClassification(BaseModel):

    category: Literal[
        "CLAIM",
        "POLICY",
        "GENERAL"
    ] = Field(
        description="Category of the customer query"
    )

    priority: Literal[
        "LOW",
        "MEDIUM",
        "HIGH"
    ] = Field(
        description="Priority of the customer query"
    )

    requires_followup: bool = Field(
        description="Whether the customer needs further assistance"
    )

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import ValidationError

json_parser = JsonOutputParser(pydantic_object=QueryClassification)

classification_prompt = ChatPromptTemplate.from_template(
    """Classify the following customer insurance query.

Customer Query: {customer_query}

{format_instructions}"""
)

classification_chain = classification_prompt | model | json_parser

result = classification_chain.invoke(
    {
        "customer_query": (
            "My claim was rejected and I need "
            "an explanation immediately."
        ),
        "format_instructions": json_parser.get_format_instructions()
    }
)

try:
    validated_result = QueryClassification.model_validate(result)

    print(validated_result)
    print(type(validated_result))

except ValidationError as error:
    print("Schema validation failed:")
    print(error)